In [ ]:
import random

# Grammar rules
rules = {
    "S": [["M", "M"], ["B", "M", "A"], ["P", "M", "R"]],
    "M": [["M", "M"], ["B", "M", "A"], ["P", "M", "R"], ["C"]],
    "B": [["clone"], ["group"]],
    "A": [["add"], ["concat"], ["matmul"]],
    "P": [["im2col"], ["permute"], ["identity"]],
    "R": [["col2im"], ["permute"], ["identity"]],
    "C": [["linear"], ["relu"], ["norm"], ["softmax"], ["identity"]],
}

# Terminals = real operations
terminals = {
    "clone","group","add","concat","matmul",
    "im2col","permute","identity","col2im",
    "linear","relu","norm","softmax"
}

def is_terminal(sym):
    return sym in terminals

def expand_once(structure):
    terminalProb = 0.32  # correct value

    for i, sym in enumerate(structure):
        if sym in rules:

            if sym == "M":
                expansion = random.choices(
                    [["M","M"], ["B","M","A"], ["P","M","R"], ["C"]],
                    weights=[(1-terminalProb)/3, (1-terminalProb)/3, (1-terminalProb)/3, terminalProb]
                )[0]
            else:
                expansion = random.choice(rules[sym])

            # ✅ RETURN MUST BE INSIDE LOOP
            return structure[:i] + expansion + structure[i+1:], sym, expansion

    return structure, None, None


def sample_with_steps():
    structure = ["S"]
    step = 0

    print(f"Step {step}: {' '.join(structure)}")

    while True:
        new_structure, expanded_sym, expansion = expand_once(structure)

        if expanded_sym is None:
            break

        step += 1
        print(f"\nStep {step}: expand {expanded_sym} → {' '.join(expansion)}")
        print(f"Result: {' '.join(new_structure)}")

        structure = new_structure

    print("\nFinal architecture:")
    print(" → ".join(structure))

    return structure   # ✅ ADD THIS





def explain_structure(structure):
    # Very simple interpretation based on patterns
    print("Explaining the Structure")
    if "clone" in structure and ("add" in structure or "concat" in structure):
        merge_op = "add" if "add" in structure else "concat"
        print("\nInterpretation:")
        print("This is a branching structure:")
        print(" - Input is split using 'clone'")
        print(" - Each branch is processed (here mostly identity/linear etc.)")
        print(f" - Branches are merged using '{merge_op}'")

        # show pseudo forward pass
        print("\nPseudo forward pass:")
        print("x1 = identity(x)")
        print("x2 = identity(x)")
        print(f"out = {merge_op}(x1, x2)")

    elif "im2col" in structure and "col2im" in structure:
        print("\nInterpretation:")
        print("This is a convolution-like block:")
        print(" - im2col extracts patches")
        print(" - linear applies filters")
        print(" - col2im restores spatial structure")

        print("\nPseudo forward pass:")
        print("patches = im2col(x)")
        print("features = linear(patches)")
        print("out = col2im(features)")

    else:
        print("\nInterpretation:")
        print("This is a simple sequential model:")
        print(" → ".join(structure))

In [ ]:
# Run once
structure = sample_with_steps()   # ✅ capture output
explain_structure(structure)      # ✅ call explanation

In [10]:
import random
from graphviz import Digraph

# Grammar rules
rules = {
    "S": [["M", "M"], ["B", "M", "A"], ["P", "M", "R"]],
    "M": [["M", "M"], ["B", "M", "A"], ["P", "M", "R"], ["C"]],
    "B": [["clone"], ["group"]],
    "A": [["add"], ["concat"], ["matmul"]],
    "P": [["im2col"], ["permute"], ["identity"]],
    "R": [["col2im"], ["permute"], ["identity"]],
    "C": [["linear"], ["relu"], ["norm"], ["softmax"], ["identity"]],
}

terminals = {
    "clone", "group", "add", "concat", "matmul",
    "im2col", "permute", "identity", "col2im",
    "linear", "relu", "norm", "softmax"
}

terminalProb = 0.32


def is_terminal(sym):
    return sym in terminals


def choose_expansion(sym):
    if sym == "M":
        return random.choices(
            [["M", "M"], ["B", "M", "A"], ["P", "M", "R"], ["C"]],
            weights=[
                (1 - terminalProb) / 3,
                (1 - terminalProb) / 3,
                (1 - terminalProb) / 3,
                terminalProb
            ]
        )[0]

    return random.choice(rules[sym])


class TreeNode:
    def __init__(self, symbol):
        self.symbol = symbol
        self.children = []


def expand_tree(node, max_depth=8, depth=0):
    """
    Recursively expands a grammar symbol into a derivation tree.
    max_depth prevents infinite recursive expansion of M.
    """

    if is_terminal(node.symbol):
        return

    if node.symbol not in rules:
        return

    # Force M to terminate if the tree gets too deep
    if node.symbol == "M" and depth >= max_depth:
        expansion = ["C"]
    else:
        expansion = choose_expansion(node.symbol)

    print(f"expand {node.symbol} → {' '.join(expansion)}")

    for sym in expansion:
        child = TreeNode(sym)
        node.children.append(child)
        expand_tree(child, max_depth=max_depth, depth=depth + 1)


def collect_terminals(node):
    """
    Returns the final architecture as a flat terminal sequence.
    """
    if is_terminal(node.symbol):
        return [node.symbol]

    result = []
    for child in node.children:
        result.extend(collect_terminals(child))

    return result


def draw_tree(root, filename="grammar_tree"):
    dot = Digraph("GrammarTree", format="pdf")

    dot.attr(rankdir="TB")
    dot.attr("node", shape="ellipse", fontname="Helvetica")
    dot.attr("edge", arrowsize="0.7")

    counter = {"id": 0}

    def add_nodes(node):
        node_id = f"n{counter['id']}"
        counter["id"] += 1

        if is_terminal(node.symbol):
            dot.node(
                node_id,
                node.symbol,
                shape="box",
                style="rounded,filled",
                fillcolor="lightgray"
            )
        else:
            dot.node(
                node_id,
                node.symbol,
                shape="circle",
                style="filled",
                fillcolor="white"
            )

        for child in node.children:
            child_id = add_nodes(child)
            dot.edge(node_id, child_id)

        return node_id

    add_nodes(root)

    dot.render(filename, cleanup=True)
    print(f"\nSaved tree as: {filename}.pdf")


def sample_and_visualize(seed=None, max_depth=8):
    if seed is not None:
        random.seed(seed)

    root = TreeNode("S")

    print("Derivation steps:\n")
    expand_tree(root, max_depth=max_depth)

    final_architecture = collect_terminals(root)

    print("\nFinal architecture:")
    print(" → ".join(final_architecture))

    draw_tree(root, filename="grammar_tree")

    return final_architecture, root


# Run
architecture, tree = sample_and_visualize(seed=7, max_depth=6)

Derivation steps:

expand S → B M A
expand B → clone
expand M → B M A
expand B → clone
expand M → M M
expand M → P M R
expand P → permute
expand M → P M R
expand P → identity
expand M → M M
expand M → C
expand C → linear
expand M → C
expand C → softmax
expand R → permute
expand R → col2im
expand M → B M A
expand B → group
expand M → M M
expand M → P M R
expand P → im2col
expand M → C
expand C → identity
expand R → col2im
expand M → P M R
expand P → permute
expand M → C
expand C → linear
expand R → col2im
expand A → add
expand A → matmul
expand A → add

Final architecture:
clone → clone → permute → identity → linear → softmax → permute → col2im → group → im2col → identity → col2im → permute → linear → col2im → add → matmul → add

Saved tree as: grammar_tree.pdf


# Grammar and Network Experiments
The bottom cells attempt to construct the NN from the grammar, however it may not be correct. The branching and meaning of everything may not be correct and has to be verified.

In [9]:
import random
from graphviz import Digraph
from pypdf import PdfWriter


# ============================================================
# Grammar rules
# ============================================================

rules = {
    "S": [["M", "M"], ["B", "M", "A"], ["P", "M", "R"]],
    "M": [["M", "M"], ["B", "M", "A"], ["P", "M", "R"], ["C"]],
    "B": [["clone"], ["group"]],
    "A": [["add"], ["concat"], ["matmul"]],
    "P": [["im2col"], ["permute"], ["identity"]],
    "R": [["col2im"], ["permute"], ["identity"]],
    "C": [["linear"], ["relu"], ["norm"], ["softmax"], ["identity"]],
}

terminals = {
    "clone", "group", "add", "concat", "matmul",
    "im2col", "permute", "identity", "col2im",
    "linear", "relu", "norm", "softmax"
}

terminalProb = 0.32


# ============================================================
# Basic utilities
# ============================================================

def is_terminal(sym):
    return sym in terminals


def choose_expansion(sym):
    """
    Chooses one production rule for a nonterminal.
    M is handled separately because it has controlled terminal probability.
    """

    if sym == "M":
        return random.choices(
            [["M", "M"], ["B", "M", "A"], ["P", "M", "R"], ["C"]],
            weights=[
                (1 - terminalProb) / 3,
                (1 - terminalProb) / 3,
                (1 - terminalProb) / 3,
                terminalProb
            ]
        )[0]

    return random.choice(rules[sym])


class TreeNode:
    def __init__(self, symbol):
        self.symbol = symbol
        self.children = []


# ============================================================
# Tree generation
# ============================================================

def expand_tree(node, max_depth=8, depth=0):
    """
    Recursively expands a grammar symbol into a derivation tree.
    max_depth prevents infinite recursive expansion of M.
    """

    if is_terminal(node.symbol):
        return

    if node.symbol not in rules:
        return

    # Force M to terminate if max depth is reached
    if node.symbol == "M" and depth >= max_depth:
        expansion = ["C"]
    else:
        expansion = choose_expansion(node.symbol)

    print(f"expand {node.symbol} -> {' '.join(expansion)}")

    for sym in expansion:
        child = TreeNode(sym)
        node.children.append(child)
        expand_tree(child, max_depth=max_depth, depth=depth + 1)


def collect_terminals(node):
    """
    Collects the final terminal sequence from the derivation tree.
    """

    if is_terminal(node.symbol):
        return [node.symbol]

    result = []

    for child in node.children:
        result.extend(collect_terminals(child))

    return result


# ============================================================
# Page 1: derivation tree
# ============================================================

def draw_derivation_tree(root):
    """
    Creates Graphviz Digraph for the derivation tree.
    """

    dot = Digraph("DerivationTree", format="pdf")

    dot.attr(rankdir="TB")
    dot.attr(size="8,10")
    dot.attr(ratio="compress")
    dot.attr("graph", margin="0.3")
    dot.attr("node", fontname="Helvetica", fontsize="10")
    dot.attr("edge", arrowsize="0.6")

    counter = {"id": 0}

    def add_nodes(node):
        node_id = f"dt_{counter['id']}"
        counter["id"] += 1

        if is_terminal(node.symbol):
            dot.node(
                node_id,
                node.symbol,
                shape="box",
                style="rounded,filled",
                fillcolor="lightgray"
            )
        else:
            dot.node(
                node_id,
                node.symbol,
                shape="circle",
                style="filled",
                fillcolor="white"
            )

        for child in node.children:
            child_id = add_nodes(child)
            dot.edge(node_id, child_id)

        return node_id

    add_nodes(root)

    return dot


# ============================================================
# Page 2: approximate computation graph from final terminals
# ============================================================

def draw_network_from_terminals(terminals_seq):
    """
    Converts the final terminal sequence into an approximate computation graph.

    Important:
    This is not an exact EinSpace execution interpreter.
    It gives a readable network-like graph using simple rules:

    clone/group        -> split into two branches
    add/concat/matmul  -> merge active branches
    identity           -> pass-through
    linear/relu/norm   -> sequential operations
    im2col/col2im      -> patch extraction/restoration style operations
    """

    dot = Digraph("NetworkGraph", format="pdf")

    dot.attr(rankdir="LR")
    dot.attr(size="11,8")
    dot.attr(ratio="compress")
    dot.attr("graph", margin="0.3")
    dot.attr("node", shape="box", style="rounded", fontname="Helvetica", fontsize="10")
    dot.attr("edge", arrowsize="0.6")

    node_counter = {"id": 0}

    def new_node(label, fillcolor=None):
        node_id = f"ng_{node_counter['id']}"
        node_counter["id"] += 1

        if fillcolor is None:
            dot.node(node_id, label)
        else:
            dot.node(
                node_id,
                label,
                style="rounded,filled",
                fillcolor=fillcolor
            )

        return node_id

    input_node = new_node("Input", fillcolor="white")
    active_paths = [input_node]

    for op in terminals_seq:

        if op in ["clone", "group"]:
            split_node = new_node(op, fillcolor="lightgray")

            for path in active_paths:
                dot.edge(path, split_node)

            branch_1 = new_node("branch 1")
            branch_2 = new_node("branch 2")

            dot.edge(split_node, branch_1)
            dot.edge(split_node, branch_2)

            active_paths = [branch_1, branch_2]

        elif op in ["add", "concat", "matmul"]:
            merge_node = new_node(op, fillcolor="lightgray")

            for path in active_paths:
                dot.edge(path, merge_node)

            active_paths = [merge_node]

        elif op == "identity":
            new_paths = []

            for path in active_paths:
                identity_node = new_node("identity")
                dot.edge(path, identity_node)
                new_paths.append(identity_node)

            active_paths = new_paths

        else:
            new_paths = []

            for path in active_paths:
                op_node = new_node(op)
                dot.edge(path, op_node)
                new_paths.append(op_node)

            active_paths = new_paths

    output_node = new_node("Output", fillcolor="white")

    for path in active_paths:
        dot.edge(path, output_node)

    return dot


# ============================================================
# Print readable approximate network
# ============================================================

def print_network_description(terminals_seq):
    """
    Prints a readable pseudo-forward interpretation of the final terminal sequence.
    """

    active = ["x"]

    print("\nApproximate network connection:\n")

    for step, op in enumerate(terminals_seq, start=1):

        if op in ["clone", "group"]:
            print(f"{step}. {op}: split {active[0]} into two branches")
            active = [f"{active[0]}_1", f"{active[0]}_2"]

        elif op in ["add", "concat", "matmul"]:
            merged = f"{op}({', '.join(active)})"
            print(f"{step}. {op}: merge branches -> {merged}")
            active = [merged]

        elif op == "identity":
            active = [f"identity({x})" for x in active]
            print(f"{step}. identity: pass through -> {', '.join(active)}")

        else:
            active = [f"{op}({x})" for x in active]
            print(f"{step}. {op}: apply operation -> {', '.join(active)}")

    print(f"\nFinal output: {active[0]}")


# ============================================================
# Merge both pages into one PDF
# ============================================================

def save_two_page_pdf(derivation_dot, network_dot, output_pdf="grammar_and_network.pdf"):
    """
    Renders both graphs separately, then merges them into a single two-page PDF.
    """

    derivation_file = "page_1_derivation_tree"
    network_file = "page_2_network_graph"

    derivation_dot.render(derivation_file, cleanup=True)
    network_dot.render(network_file, cleanup=True)

    writer = PdfWriter()

    writer.append(f"{derivation_file}.pdf")
    writer.append(f"{network_file}.pdf")

    with open(output_pdf, "wb") as f:
        writer.write(f)

    print(f"\nSaved combined PDF as: {output_pdf}")


# ============================================================
# Main function
# ============================================================

def sample_and_visualize(seed=None, max_depth=8):
    """
    Generates:
    1. A derivation tree
    2. A final terminal architecture
    3. An approximate computation graph
    4. A two-page PDF containing both graphs
    """

    if seed is not None:
        random.seed(seed)

    root = TreeNode("S")

    print("Derivation steps:\n")
    expand_tree(root, max_depth=max_depth)

    final_architecture = collect_terminals(root)

    print("\nFinal architecture:")
    print(" -> ".join(final_architecture))

    print_network_description(final_architecture)

    derivation_dot = draw_derivation_tree(root)
    network_dot = draw_network_from_terminals(final_architecture)

    save_two_page_pdf(
        derivation_dot,
        network_dot,
        output_pdf="grammar_and_network.pdf"
    )

    return final_architecture, root


# ============================================================
# Run
# ============================================================

if __name__ == "__main__":
    architecture, tree = sample_and_visualize(
        seed=7,
        max_depth=6
    )

Derivation steps:

expand S -> B M A
expand B -> clone
expand M -> B M A
expand B -> clone
expand M -> M M
expand M -> P M R
expand P -> permute
expand M -> P M R
expand P -> identity
expand M -> M M
expand M -> C
expand C -> linear
expand M -> C
expand C -> softmax
expand R -> permute
expand R -> col2im
expand M -> B M A
expand B -> group
expand M -> M M
expand M -> P M R
expand P -> im2col
expand M -> C
expand C -> identity
expand R -> col2im
expand M -> P M R
expand P -> permute
expand M -> C
expand C -> linear
expand R -> col2im
expand A -> add
expand A -> matmul
expand A -> add

Final architecture:
clone -> clone -> permute -> identity -> linear -> softmax -> permute -> col2im -> group -> im2col -> identity -> col2im -> permute -> linear -> col2im -> add -> matmul -> add

Approximate network connection:

1. clone: split x into two branches
2. clone: split x_1 into two branches
3. permute: apply operation -> permute(x_1_1), permute(x_1_2)
4. identity: pass through -> identity(permu

In [6]:
import random
from graphviz import Digraph
from pypdf import PdfWriter


# -----------------------------
# Grammar rules
# -----------------------------
rules = {
    "S": [["M", "M"], ["B", "M", "A"], ["P", "M", "R"]],
    "M": [["M", "M"], ["B", "M", "A"], ["P", "M", "R"], ["C"]],
    "B": [["clone"], ["group"]],
    "A": [["add"], ["concat"], ["matmul"]],
    "P": [["im2col"], ["permute"], ["identity"]],
    "R": [["col2im"], ["permute"], ["identity"]],
    "C": [["linear"], ["relu"], ["norm"], ["softmax"], ["identity"]],
}

# Terminals = real operations
terminals = {
    "clone", "group", "add", "concat", "matmul",
    "im2col", "permute", "identity", "col2im",
    "linear", "relu", "norm", "softmax"
}

terminalProb = 0.32


# -----------------------------
# Helper functions
# -----------------------------
def is_terminal(sym):
    return sym in terminals


def choose_expansion(sym):
    """
    Chooses one expansion rule for a nonterminal.
    M uses the special terminal probability.
    Other nonterminals are sampled uniformly.
    """

    if sym == "M":
        return random.choices(
            [["M", "M"], ["B", "M", "A"], ["P", "M", "R"], ["C"]],
            weights=[
                (1 - terminalProb) / 3,
                (1 - terminalProb) / 3,
                (1 - terminalProb) / 3,
                terminalProb
            ]
        )[0]

    return random.choice(rules[sym])


# -----------------------------
# Tree node class
# -----------------------------
class TreeNode:
    def __init__(self, symbol):
        self.symbol = symbol
        self.children = []


# -----------------------------
# Build derivation tree
# -----------------------------
def expand_tree(node, max_depth=8, depth=0):
    """
    Recursively expands a grammar symbol into a derivation tree.
    max_depth prevents infinite recursive expansion of M.
    """

    if is_terminal(node.symbol):
        return

    if node.symbol not in rules:
        return

    # Force M to terminate if the tree gets too deep
    if node.symbol == "M" and depth >= max_depth:
        expansion = ["C"]
    else:
        expansion = choose_expansion(node.symbol)

    print(f"expand {node.symbol} -> {' '.join(expansion)}")

    for sym in expansion:
        child = TreeNode(sym)
        node.children.append(child)
        expand_tree(child, max_depth=max_depth, depth=depth + 1)


def collect_terminals(node):
    """
    Returns the final architecture as a flat terminal sequence.
    """

    if is_terminal(node.symbol):
        return [node.symbol]

    result = []

    for child in node.children:
        result.extend(collect_terminals(child))

    return result


# -----------------------------
# Page 1: Draw derivation tree
# -----------------------------
def draw_derivation_tree(root, filename="page1_derivation_tree"):
    dot = Digraph("DerivationTree", format="pdf")

    dot.attr(rankdir="TB")
    dot.attr("node", shape="ellipse", fontname="Helvetica")
    dot.attr("edge", arrowsize="0.7")

    counter = {"id": 0}

    def add_nodes(node):
        node_id = f"n{counter['id']}"
        counter["id"] += 1

        if is_terminal(node.symbol):
            dot.node(
                node_id,
                node.symbol,
                shape="box",
                style="rounded,filled",
                fillcolor="lightgray"
            )
        else:
            dot.node(
                node_id,
                node.symbol,
                shape="circle",
                style="filled",
                fillcolor="white"
            )

        for child in node.children:
            child_id = add_nodes(child)
            dot.edge(node_id, child_id)

        return node_id

    add_nodes(root)

    pdf_path = dot.render(filename, cleanup=True)
    return pdf_path


# -----------------------------
# Page 2: Convert derivation tree to network graph
# -----------------------------
def get_terminal_child(node):
    """
    For nodes like B, A, P, R, C, return their terminal child.

    Example:
    B -> clone
    A -> add
    C -> linear
    """

    if len(node.children) == 1 and is_terminal(node.children[0].symbol):
        return node.children[0].symbol

    return node.symbol


def draw_network(root, filename="page2_network_graph"):
    dot = Digraph("NetworkGraph", format="pdf")

    dot.attr(rankdir="LR")
    dot.attr("node", shape="box", style="rounded,filled", fillcolor="lightgray", fontname="Helvetica")
    dot.attr("edge", arrowsize="0.7")

    counter = {"id": 0}

    def new_node(label, fillcolor="lightgray"):
        node_id = f"op{counter['id']}"
        counter["id"] += 1

        dot.node(
            node_id,
            label,
            fillcolor=fillcolor
        )

        return node_id

    input_node = new_node("input", fillcolor="white")

    def build_network(node, incoming_node):
        """
        Converts the derivation tree into a possible network graph.

        Interpretation used here:

        M M
            Sequential composition.

        P M R
            Preprocess, apply module, then restore.

        B M A
            Split input, send one branch through M,
            keep one skip branch, then merge using A.

        C
            Real operation such as linear, relu, norm, softmax, identity.
        """

        # Direct terminal operation
        if is_terminal(node.symbol):
            op_node = new_node(node.symbol)
            dot.edge(incoming_node, op_node)
            return op_node

        child_symbols = [child.symbol for child in node.children]

        # S -> M M or M -> M M
        if child_symbols == ["M", "M"]:
            out1 = build_network(node.children[0], incoming_node)
            out2 = build_network(node.children[1], out1)
            return out2

        # S -> P M R or M -> P M R
        if child_symbols == ["P", "M", "R"]:
            p_op = get_terminal_child(node.children[0])
            r_op = get_terminal_child(node.children[2])

            p_node = new_node(p_op)
            dot.edge(incoming_node, p_node)

            middle_out = build_network(node.children[1], p_node)

            r_node = new_node(r_op)
            dot.edge(middle_out, r_node)

            return r_node

        # S -> B M A or M -> B M A
        if child_symbols == ["B", "M", "A"]:
            b_op = get_terminal_child(node.children[0])
            a_op = get_terminal_child(node.children[2])

            split_node = new_node(b_op)
            dot.edge(incoming_node, split_node)

            skip_node = new_node("skip", fillcolor="white")
            dot.edge(split_node, skip_node)

            branch_out = build_network(node.children[1], split_node)

            merge_node = new_node(a_op)
            dot.edge(skip_node, merge_node)
            dot.edge(branch_out, merge_node)

            return merge_node

        # M -> C
        if child_symbols == ["C"]:
            return build_network(node.children[0], incoming_node)

        # C -> terminal
        if len(node.children) == 1 and is_terminal(node.children[0].symbol):
            op_node = new_node(node.children[0].symbol)
            dot.edge(incoming_node, op_node)
            return op_node

        # Fallback, mostly for safety
        current_node = incoming_node

        for child in node.children:
            current_node = build_network(child, current_node)

        return current_node

    output_node = build_network(root, input_node)

    final_node = new_node("output", fillcolor="white")
    dot.edge(output_node, final_node)

    pdf_path = dot.render(filename, cleanup=True)
    return pdf_path


# -----------------------------
# Merge both PDFs into one file
# -----------------------------
def merge_pdfs(pdf_paths, output_pdf="grammar_tree_with_network.pdf"):
    writer = PdfWriter()

    for pdf in pdf_paths:
        writer.append(pdf)

    with open(output_pdf, "wb") as f:
        writer.write(f)

    print(f"\nSaved combined PDF as: {output_pdf}")


# -----------------------------
# Main function
# -----------------------------
def sample_and_visualize(seed=None, max_depth=8):
    if seed is not None:
        random.seed(seed)

    root = TreeNode("S")

    print("Derivation steps:\n")
    expand_tree(root, max_depth=max_depth)

    final_architecture = collect_terminals(root)

    print("\nFinal architecture:")
    print(" -> ".join(final_architecture))

    derivation_pdf = draw_derivation_tree(
        root,
        filename="page1_derivation_tree"
    )

    network_pdf = draw_network(
        root,
        filename="page2_network_graph"
    )

    merge_pdfs(
        [derivation_pdf, network_pdf],
        output_pdf="grammar_tree_with_network.pdf"
    )

    return final_architecture, root


# -----------------------------
# Run
# -----------------------------
architecture, tree = sample_and_visualize( max_depth=6)

Derivation steps:

expand S -> P M R
expand P -> im2col
expand M -> M M
expand M -> B M A
expand B -> group
expand M -> M M
expand M -> P M R
expand P -> im2col
expand M -> C
expand C -> linear
expand R -> identity
expand M -> P M R
expand P -> permute
expand M -> B M A
expand B -> group
expand M -> C
expand C -> identity
expand A -> concat
expand R -> identity
expand A -> concat
expand M -> M M
expand M -> M M
expand M -> B M A
expand B -> clone
expand M -> M M
expand M -> C
expand C -> norm
expand M -> C
expand C -> identity
expand A -> matmul
expand M -> C
expand C -> norm
expand M -> C
expand C -> norm
expand R -> col2im

Final architecture:
im2col -> group -> im2col -> linear -> identity -> permute -> group -> identity -> concat -> identity -> concat -> clone -> norm -> identity -> matmul -> norm -> norm -> col2im

Saved combined PDF as: grammar_tree_with_network.pdf
